MCP SCENARIO: “Smart Banking Support Assistant”
🧩 Scenario Background
You are working in a company called FinTrust Bank.
Customers often face issues such as:
- Credit card not working
- Trouble with online banking login
- Queries about loan status
- Transaction disputes
👉 Instead of calling customer care, customers use an AI Banking Support Bot.

🤖 What this Bot Should Do
When a customer types a problem:
- Understand the issue (e.g., “My card was declined”)
- Decide if escalation to a human agent is needed
- Identify:
- Category (Card Services / Online Banking / Loans / Transactions)
- Priority (High / Medium)
- Create a support ticket if required
- Provide instant guidance (FAQs, troubleshooting steps, policy info)
- Show confirmation and next steps

🧠 How MCP Fits Here
|  |  |
|  |  |
|  |  |
|  |  |
|  |  |



This way, MCP is applied in a financial services context, where the AI assistant reduces call center load, provides quick resolutions, and ensures customers feel supported with secure, reliable guidance.
Would you like me to craft one more in a healthcare setting (like hospital patient support), so you can see how MCP adapts to critical service environments?

In [ ]:
# ============================================
# MCP SCENARIO: Smart Banking Support Assistant
# WITHOUT API / WITHOUT LLM
# ============================================

from datetime import datetime

# ============================================
# STEP 1: DATABASE
# ============================================

tickets_db = []

# ============================================
# STEP 2: TOOL LAYER
# ============================================

def create_support_ticket(issue, priority, category, context):
    ticket_id = f"BNK{1000 + len(tickets_db)}"

    ticket = {
        "ticket_id": ticket_id,
        "issue": issue,
        "priority": priority,
        "category": category,
        "created_at": datetime.now().isoformat(),
        "source": "banking_support_bot",
        "context": context
    }

    tickets_db.append(ticket)
    return ticket

# ============================================
# STEP 3: CONTEXT OBJECT
# ============================================

def build_context(user_input):
    return {
        "user_input": user_input,
        "session_id": f"session_{len(tickets_db) + 1}",
        "timestamp": datetime.now().isoformat(),
        "agent_name": "BankingSupportAgent"
    }

# ============================================
# STEP 4: GUIDANCE LAYER
# ============================================

def provide_guidance(category):
    faq_map = {
        "card services": (
            "Try these steps:\n"
            "1. Check if your card is active and not expired\n"
            "2. Verify the correct PIN or OTP\n"
            "3. Check if online/international transactions are enabled\n"
            "4. If the card is still declined, support escalation is needed"
        ),
        "online banking": (
            "Try these steps:\n"
            "1. Verify your username and password\n"
            "2. Use 'Forgot Password' if login fails\n"
            "3. Check OTP/mobile number linkage\n"
            "4. If access is still blocked, support is needed"
        ),
        "loans": (
            "Try these steps:\n"
            "1. Keep your loan application/reference number ready\n"
            "2. Check the loan status in the bank portal/app\n"
            "3. Review any SMS/email updates\n"
            "4. If status is delayed or unclear, support can help"
        ),
        "transactions": (
            "Try these steps:\n"
            "1. Check recent transaction history\n"
            "2. Verify if the transaction is pending or completed\n"
            "3. Check merchant details and alerts\n"
            "4. If it looks incorrect, raise a dispute ticket"
        ),
        "general": (
            "Please share a bit more detail about your banking issue so I can guide you properly."
        )
    }
    return faq_map.get(category, faq_map["general"])

# ============================================
# STEP 5: ANALYSIS LAYER (RULE-BASED)
# ============================================

def analyze_issue(user_input):
    text = user_input.lower()

    # Category detection
    if "card" in text or "credit card" in text or "debit card" in text or "declined" in text:
        category = "card services"
    elif "login" in text or "online banking" in text or "password" in text or "otp" in text:
        category = "online banking"
    elif "loan" in text or "emi" in text or "application" in text:
        category = "loans"
    elif "transaction" in text or "dispute" in text or "payment" in text or "charged" in text:
        category = "transactions"
    else:
        category = "general"

    # Priority detection
    if "urgent" in text or "immediately" in text or "blocked" in text or "cannot" in text or "can't" in text:
        priority = "high"
    else:
        priority = "medium"

    # Ticket creation decision
    escalation_keywords = [
        "declined", "blocked", "cannot", "can't", "failed",
        "dispute", "fraud", "not working", "urgent", "error"
    ]

    create_ticket = any(word in text for word in escalation_keywords)

    # Reason
    if create_ticket:
        short_reason = "Issue appears operational or blocking and needs support follow-up"
    else:
        short_reason = "Issue seems informational or solvable with guidance"

    return {
        "create_ticket": create_ticket,
        "category": category,
        "priority": priority,
        "short_reason": short_reason
    }

# ============================================
# STEP 6: EXECUTION METADATA
# ============================================

def build_metadata(context, decision):
    return {
        "agent_name": context["agent_name"],
        "session_id": context["session_id"],
        "timestamp": datetime.now().isoformat(),
        "decision_summary": decision["short_reason"]
    }

# ============================================
# STEP 7: MCP ORCHESTRATOR
# ============================================

def mcp_banking_agent(user_input):
    context = build_context(user_input)
    print("\n🧠 Agent received:", user_input)
    print("🗂️ Context:", context)

    decision = analyze_issue(user_input)
    print("🤖 Decision:", decision)

    guidance = provide_guidance(decision["category"])

    metadata = build_metadata(context, decision)
    print("📝 Metadata:", metadata)

    if decision["create_ticket"]:
        payload = {
            "issue": user_input,
            "priority": decision["priority"],
            "category": decision["category"],
            "context": context
        }

        print("📦 MCP Payload:", payload)

        result = create_support_ticket(**payload)

        return f"""
✅ Support Ticket Created Successfully!

Ticket ID: {result['ticket_id']}
Issue: {result['issue']}
Category: {result['category']}
Priority: {result['priority']}
Created At: {result['created_at']}

Why ticket was created:
- {decision['short_reason']}

Instant Guidance:
{guidance}

Next Step:
- Our banking support team will review this case shortly.
- Keep your ticket ID for future follow-up.
"""

    else:
        return f"""
🤖 No Ticket Required Right Now

Reason:
- {decision['short_reason']}

Suggested Guidance:
{guidance}

Next Step:
- Try the above banking guidance first.
- If the issue continues, raise the issue again with more detail.
"""

# ============================================
# STEP 8: RUN LOOP
# ============================================

print("🚀 Banking Support Assistant Started (type 'exit')\n")

while True:
    user_input = input("Enter banking issue: ").strip()

    if user_input.lower() == "exit":
        print("👋 Exiting...")
        break

    if not user_input:
        print("⚠️ Please enter a valid issue.")
        continue

    response = mcp_banking_agent(user_input)
    print(response)

🚀 Banking Support Assistant Started (type 'exit')

Enter banking issue: What is the status of my loan application?

🧠 Agent received: What is the status of my loan application?
🗂️ Context: {'user_input': 'What is the status of my loan application?', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:52:37.274900', 'agent_name': 'BankingSupportAgent'}
🤖 Decision: {'create_ticket': False, 'category': 'loans', 'priority': 'medium', 'short_reason': 'Issue seems informational or solvable with guidance'}
📝 Metadata: {'agent_name': 'BankingSupportAgent', 'session_id': 'session_1', 'timestamp': '2026-03-28T06:52:37.275069', 'decision_summary': 'Issue seems informational or solvable with guidance'}

🤖 No Ticket Required Right Now

Reason:
- Issue seems informational or solvable with guidance

Suggested Guidance:
Try these steps:
1. Keep your loan application/reference number ready
2. Check the loan status in the bank portal/app
3. Review any SMS/email updates
4. If status is delayed or uncl